In [1]:
import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim


import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

In [2]:
torch.cuda.is_available()

True

In [9]:
print(torch.version.cuda)

None


In [3]:
import torchvision.models as models

In [4]:
weights = models.ResNet18_Weights.IMAGENET1K_V1

In [32]:
transform = weights.transforms()

In [6]:
model = models.resnet18(weights = weights)

In [33]:
training_data = ImageFolder(root = "./data/train", transform = transform)

In [34]:
training_data.classes

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [35]:
training_loader = DataLoader(training_data, batch_size = 32, shuffle = True)

In [9]:
for param in model.parameters():
    param.requires_grad = False

In [10]:
model.fc.in_features

512

In [11]:
model.fc = nn.Linear(model.fc.in_features, 7)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr = 0.001)

In [13]:
losses = []
scores = {}
epoch_count = 0

In [ ]:
#Training Loop
epochs = 1

for epoch in range(epochs):
    model.train()
    corrects = 0
    wrongs = 0
    running_loss = 0
    
    for b, (X_train, y_train) in enumerate(training_loader):
        batch = b+1
        
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)

        running_loss += loss.item()
        predicted = torch.argmax(y_pred, dim = 1)

        corrects += (predicted == y_train).sum().item()
        wrongs += (predicted != y_train).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if b%32 == 0:
            print(f"Epoch: {epoch}/{epochs} Batch: {batch} Current Loss: {running_loss/batch}")
            

    epoch_count += 1
    print(f"Epoch: {epoch}. Current Accuracy: {corrects/(wrongs+corrects)}")
    losses.append(running_loss/batch)


        